In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import os
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, GRU
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error
np.random.seed(42)
tf.random.set_seed(42)
from sklearn.model_selection import KFold
os.environ['PYTHONHASHSEED'] = str(42)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '' 
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import TimeSeriesSplit

In [ ]:
# pre process 1 separa o dataset em intervalo e depois retorna o df com a maior sequencia sem falhas
def pre_process_timeseries(intervalo, caminho):
    df = pd.read_csv(caminho)
    df['timestamp'] = pd.to_datetime(df['Timestamp_cubic'], unit='s')
    df.set_index('timestamp', inplace=True)
    df = df.resample(intervalo).agg({
        'Vazao': 'mean',       
         'Vazao_bbr': 'mean', 
        'Hop_count': 'mean',  
        'Atraso(ms)': 'mean', 
        'Bottleneck': 'min'        
    })
    df.reset_index(inplace=True)
    df.set_index('timestamp', inplace=True)

    #procurando a maior sequencia sem falhas
    df['sequence'] = df.notna().all(axis=1)
    max_sequence = []
    temp_sequence = []

    for i, sequence in enumerate(df['sequence']):
        if sequence:
            temp_sequence.append(i)
        else:
            if len(temp_sequence) > len(max_sequence):
                max_sequence = temp_sequence
            temp_sequence = []
    if len(temp_sequence) > len(max_sequence):
        max_sequence = temp_sequence
    df = df.iloc[max_sequence].drop(columns=['sequence']).reset_index(drop=True)
    return df, len(max_sequence)

In [ ]:
# Usando os dados sem separar em intervalos, e timestamp como uma feature
def pre_process_notimeseries(caminho):
    df = pd.read_csv(caminho)
    df['Timestamp_cubic'] = pd.to_datetime(df['Timestamp_cubic'])  
    df.set_index('Timestamp_cubic', inplace=True)  
    df = df.drop(columns=['Link_bottleneck'])
    return df

In [ ]:
#serie temporal  - validação cruzada
def prepare_data(df, target_col, n_steps):
    X, y = [], []
    for i in range(len(df) - n_steps):

        X.append(df.iloc[i:i + n_steps].drop(columns=[target_col]).values)  
        y.append(df[target_col].iloc[i + n_steps])
    return np.array(X), np.array(y)


#serie temporal
def time_series_cross_validation(df, target_col, model, n_steps, n_splits):
    rmse_scores = []
    split_size = len(df) // n_splits
    scaler = MinMaxScaler()
    df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

    # tscv = TimeSeriesSplit(n_splits=n_splits)
    # for train_index, test_index in tscv.split(df_scaled):
    #     train_df, test_df = df_scaled[train_index], df_scaled[test_index]
    #     X_train, y_train = prepare_data(train_df, target_col, n_steps)
    #     X_test, y_test = prepare_data(test_df, target_col, n_steps)
    
    for i in range(1, n_splits + 1):
        train_df = df_scaled[:split_size * i]
        test_df = df_scaled[split_size * i : split_size * (i + 1)]
        X_train, y_train = prepare_data(train_df, target_col, n_steps)
        X_test, y_test = prepare_data(test_df, target_col, n_steps)

        if len(X_test) == 0:
            break
        model = create_lstm(64, X_train, 0.00001)
        early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)
        model.fit(X_train, y_train, epochs=100, batch_size=32,
                  validation_data=(X_test, y_test), callbacks=[early_stop], verbose=10)
        y_pred = model.predict(X_test)

        y_pred_rescaled = scaler.inverse_transform(
            np.column_stack([y_pred, np.zeros((y_pred.shape[0], df.shape[1] - 1))]))[:, 0]
        y_test_rescaled = scaler.inverse_transform(
            np.column_stack([y_test, np.zeros((y_test.shape[0], df.shape[1] - 1))]))[:, 0]

        rmse = np.sqrt(mean_squared_error(y_test_rescaled, y_pred_rescaled))
        rmse_scores.append(rmse)
    return np.mean(rmse_scores)/1000000, y_test_rescaled, y_pred_rescaled


# no time series, usando timestamp como feature
def cross_validate_lstm(df, target_col, model, n_steps, n_splits=5):
    scaler = MinMaxScaler()
    df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
    X, y = prepare_data(df_scaled, target_col, n_steps)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    rmse_scores = []

    # Validação cruzada
    for train_index, test_index in kf.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        model = create_lstm(64, X_train, 0.00001)
        early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)
        model.fit(X_train, y_train, epochs=100, batch_size=32,
                  validation_data=(X_test, y_test), callbacks=[early_stop], verbose=10)

        y_pred = model.predict(X_test)
        y_pred_rescaled = scaler.inverse_transform(
            np.column_stack([y_pred, np.zeros((y_pred.shape[0], df.shape[1] - 1))]))[:, 0]
        y_test_rescaled = scaler.inverse_transform(
            np.column_stack([y_test, np.zeros((y_test.shape[0], df.shape[1] - 1))]))[:, 0]
        rmse = np.sqrt(mean_squared_error(y_test_rescaled, y_pred_rescaled))
        rmse_scores.append(rmse)
        print(f"Fold RMSE: {rmse}")
    print("RMSE médio na validação cruzada:", np.mean(rmse_scores))
    return np.mean(rmse_scores)/1000000, y_test_rescaled, y_pred_rescaled


def create_gru(units, train, learning_rate): 
    model = Sequential() 
    model.add(GRU(units = units, return_sequences = True, input_shape = [train.shape[1], train.shape[2]]))
    model.add(GRU(units = units))   
    model.add(Dense(1))
    model.compile(loss=MeanSquaredError(), optimizer = Adam(learning_rate=learning_rate), metrics=[RootMeanSquaredError()])
   
    return model

# Create LSTM model
def create_lstm(units, train, learning_rate): 
    model = Sequential() 
    model.add(LSTM(units = units, return_sequences = True, input_shape = [train.shape[1], train.shape[2]]))
    model.add(LSTM(units = units)) 
    model.add(Dense(1))
    model.compile(loss=MeanSquaredError(), optimizer = Adam(learning_rate=learning_rate), metrics=[RootMeanSquaredError()])
    
    return model

In [ ]:
root = '../arquivo-completo/'
intervals = ['4h', '6h']
targets = ['Vazao', 'Vazao_bbr']
arquivo_saida = '../arquivo-completo/resultados_rmse_timeseries.txt'
with open(arquivo_saida, 'a') as f:
    for arquivo in os.listdir(root):
        if arquivo.endswith('.csv'):
            caminho_completo = os.path.join(root, arquivo)
            name_arq = arquivo.split(' ')[5].split('.')[0]
            for interval in intervals:
                df, len_arq = pre_process_timeseries(interval, caminho_completo)
                for target in targets: 
                    rmse, y_test, y_pred = time_series_cross_validation(df, target, create_lstm, 10, 5)
                    f.write(f'Variavel: {target}, Arquivo: {name_arq}, Intervalo: {interval}, RMSE: {rmse}, Size: {len_arq}\n')
                #print(f'Arquivo: {name_arq}, Intervalo: {interval}, RMSE: {rmse}')  

In [ ]:
root = '../arquivo-completo/'
arquivo_saida = '../arquivo-completo/resultados_rmse_notimeseries.txt'
targets = ['Vazao', 'Vazao_bbr']
with open(arquivo_saida, 'a') as f:
    for arquivo in os.listdir(root):
        if arquivo.endswith('.csv'):
            caminho_completo = os.path.join(root, arquivo)
            name_arq = arquivo.split(' ')[5].split('.')[0]
            df = pre_process_notimeseries(caminho_completo)
            for target in targets:
                rmse, y_test, y_pred = cross_validate_lstm(df, target, create_lstm, 10, 5)
                f.write(f'Alvo: {target}, Arquivo: {name_arq}, RMSE: {rmse}\n')
            #print(f'Arquivo: {name_arq}, RMSE: {rmse}') 

In [ ]:
def plot_predictions(y_test, y_pred):
    plt.figure(figsize=(14, 7))
    plt.plot(y_test, label='Actual Values', color='blue', linewidth=2)
    plt.plot(y_pred, label='Predicted Values', color='orange', linestyle='--', linewidth=2)
    plt.title('Actual vs Predicted Values', fontsize=16)
    plt.xlabel('Time Steps', fontsize=14)
    plt.ylabel('Values', fontsize=14)
    plt.legend()
    plt.grid()
    plt.show()
    
um = y_test[:200]
dois = y_pred[:200]
plot_predictions(um, dois)
